# Creates table S1

In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import matthews_corrcoef, confusion_matrix
from scipy.stats import fisher_exact

In [2]:
path = ("../Results/ctrl_final_combined_results.csv'")

In [3]:
table = pd.read_csv(path)

## Summarize data (ALL data, NOT per feature)

In [14]:
from scipy.stats import fisher_exact

def summarize_all_data_cutoffs(data):
    cell_lines = ['HepG2', 'K562']
    shap_cutoffs = [0, 0.05, 0.1, 0.2]
    rows = []
    for cell_line in cell_lines:
        for cutoff in shap_cutoffs:
            # ------------------------------------------------------------------
            # 1. Subset & significance filter
            # ------------------------------------------------------------------
            subset = data[data['cell_line'] == cell_line].copy()
            subset['Significance'] = (
                (subset['dPSI'].abs() > 0) &
                (subset['FDR'] <= 0.1) &
                (subset['CTRL_SHAP'].abs() >= cutoff)
            ).astype(int)
            significant_subset = subset[subset['Significance'] == 1].copy()
            num_significant = len(significant_subset)
            
            # ------------------------------------------------------------------
            # 2. MCC + Fisher's Exact Test
            # ------------------------------------------------------------------
            y_true = np.sign(significant_subset['dPSI'].values).astype(int)
            y_pred = np.sign(significant_subset['CTRL_SHAP'].values).astype(int)


            
            nonzero_mask = (y_true != 0) & (y_pred != 0)
            y_true_nz = y_true[nonzero_mask]
            y_pred_nz = y_pred[nonzero_mask]

            
            mcc = matthews_corrcoef(y_true_nz, y_pred_nz)
            tp = ((y_pred_nz ==  1) & (y_true_nz ==  1)).sum()
            fp = ((y_pred_nz ==  1) & (y_true_nz == -1)).sum()
            fn = ((y_pred_nz == -1) & (y_true_nz ==  1)).sum()
            tn = ((y_pred_nz == -1) & (y_true_nz == -1)).sum()
            odds_ratio, fisher_p = fisher_exact([[tp, fp], [fn, tn]], alternative='two-sided')
            
            # ------------------------------------------------------------------
            # 3. dPSI–SHAP concordance
            # ------------------------------------------------------------------
            concordance = np.mean(y_true_nz == y_pred_nz) * 100
            rows.append({
                'Cell Line': cell_line,
                'SHAP Cutoff': cutoff,
                'Significant Events': num_significant,
                'dPSI Local SHAP Concordance': concordance,
                'MCC': mcc
            })
    return pd.DataFrame(rows).sort_values(['Cell Line', 'SHAP Cutoff']).reset_index(drop=True)

In [15]:
all_data_cut = summarize_all_data_cutoffs(table)

## Summarize the data per feature (no SHAP cutoff)

In [17]:
def summarize_crispr_no_event_cut(data):
    cell_lines = ['HepG2', 'K562']
    features = data['Feature'].unique()
    rows = []
    
    for feature in features:
        parts = feature.split('_')
        rbp_name = parts[0]
        position = parts[1]
        rbp_label = f"{rbp_name} Position {position}"
        for cell_line in cell_lines:
    
            # ------------------------------------------------------------------
            # 1. Subset & significance filter
            # ------------------------------------------------------------------
            feature_subset = data[
                (data['cell_line'] == cell_line) &
                (data['Feature'] == feature)
            ].copy()
    
            feature_subset['Significance'] = (
                (feature_subset['dPSI'].abs() > 0) &
                (feature_subset['FDR'] <= 0.1)
            ).astype(int)
    
            significant_subset = feature_subset[feature_subset['Significance'] == 1].copy()
            num_significant = len(significant_subset)
    
            # ------------------------------------------------------------------
            # 2. Fisher's Exact Test — requires at least 1 significant event
            # ------------------------------------------------------------------
            if num_significant >= 1:
                y_true = np.sign(significant_subset['dPSI'].values).astype(int)
                y_pred = np.sign(significant_subset['CTRL_SHAP'].values).astype(int)

                # Drop zeros from both vectors consistently
                nonzero_mask = (y_true != 0) & (y_pred != 0)
                y_true_nz = y_true[nonzero_mask]
                y_pred_nz = y_pred[nonzero_mask]

                if len(y_true_nz) == 0:
                    concordance = np.nan
                    odds_ratio, fisher_p = np.nan, np.nan
                    tp = fp = fn = tn = np.nan
                    mcc = np.nan
               
                else:
                    tp = ((y_pred_nz ==  1) & (y_true_nz ==  1)).sum()
                    fp = ((y_pred_nz ==  1) & (y_true_nz == -1)).sum()
                    fn = ((y_pred_nz == -1) & (y_true_nz ==  1)).sum()
                    tn = ((y_pred_nz == -1) & (y_true_nz == -1)).sum()

                    # ------------------------------------------------------------------
                    # 3. PSI–SHAP concordance (% of events where signs match)
                    # ------------------------------------------------------------------
                    concordance = np.mean(y_true_nz == y_pred_nz) * 100

                    mcc = matthews_corrcoef(y_true_nz, y_pred_nz)

                    # Fisher's requires observations in at least 2 cells of the table
                    table = [[tp, fp], [fn, tn]]
                    odds_ratio, fisher_p = fisher_exact(table, alternative='two-sided')


                # Append Results to Table

                # Only include row if concordance is valid (avoids rows with sig. events but can't calculate FET/MCC
                if np.isnan(concordance):
                    continue
            
                row = {
                    'Feature': rbp_label,
                    'Cell Line': cell_line,
                    'Significant Events': num_significant,
                    'dPSI Local SHAP Concordance': concordance,
                }
            
                if not np.isnan(mcc):
                    row['MCC'] = mcc
                if not any(np.isnan(v) for v in [tp, tn, fp, fn]):
                    row['True Positives'] = tp
                    row['True Negatives'] = tn
                    row['False Negatives'] = fn
                    row['False Positives'] = fp
            
                rows.append(row)


    return pd.DataFrame(rows).sort_values('Significant Events', ascending=False).reset_index(drop=True)

In [18]:
crispr_data = summarize_crispr_no_event_cut(table)

/home/bhd5qr/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/bhd5qr/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/bhd5qr/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/bhd5qr/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matr